# 🏗️ 바닥면 평탄성 분석 (Floor Flatness Analysis)

레이저 스캐닝으로 추출된 포인트 클라우드 데이터를 활용하여 건설 현장 바닥면의 평탄성을 분석합니다.

## 분석 과정
1. 데이터 로드 및 전처리
2. 아웃라이어 제거 (Statistical Outlier Removal)
3. RANSAC을 이용한 기준 평면 피팅
4. 평면으로부터의 거리(편차) 계산
5. 평탄하지 않은 영역 하이라이트 및 시각화

## 1. 라이브러리 설치 및 임포트

In [ ]:
# 필요한 라이브러리 설치
!pip install open3d plotly -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from google.colab import files
import open3d as o3d
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from scipy.interpolate import griddata
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 로드 완료!")

## 2. 데이터 업로드 및 로드

In [ ]:
# 파일 업로드
print("📁 포인트 클라우드 파일을 업로드하세요 (.txt, .xyz, .csv)")
uploaded = files.upload()

In [ ]:
# 업로드된 파일 로드
filename = list(uploaded.keys())[0]
print(f"📁 로드된 파일: {filename}")

# 데이터 로드
try:
    data = np.loadtxt(filename)
except:
    data = pd.read_csv(filename, sep=None, engine='python', header=None).values

print(f"\n📊 데이터 정보:")
print(f"   - 포인트 수: {data.shape[0]:,}")
print(f"   - 컬럼 수: {data.shape[1]}")

# 컬럼 설명
if data.shape[1] >= 7:
    print("   - 컬럼 구성: X, Y, Z, R, G, B, Intensity")
elif data.shape[1] >= 6:
    print("   - 컬럼 구성: X, Y, Z, R, G, B")
else:
    print("   - 컬럼 구성: X, Y, Z")

# XYZ 추출
xyz = data[:, :3]

print(f"\n📏 데이터 범위:")
print(f"   X: [{xyz[:, 0].min():.3f}, {xyz[:, 0].max():.3f}] m")
print(f"   Y: [{xyz[:, 1].min():.3f}, {xyz[:, 1].max():.3f}] m")
print(f"   Z: [{xyz[:, 2].min():.3f}, {xyz[:, 2].max():.3f}] m")

## 3. 원본 포인트 클라우드 시각화

In [ ]:
def visualize_pointcloud_3d(points, colors=None, title="Point Cloud", size=1, sample_size=100000):
    """3D 포인트 클라우드 시각화 (Plotly)"""
    
    # 샘플링
    if len(points) > sample_size:
        idx = np.random.choice(len(points), sample_size, replace=False)
        points_vis = points[idx]
        colors_vis = colors[idx] if colors is not None else None
        print(f"⚠️ 시각화를 위해 {sample_size:,}개 포인트로 샘플링됨")
    else:
        points_vis = points
        colors_vis = colors
    
    if colors_vis is None:
        colors_vis = points_vis[:, 2]  # Z값으로 색상
    
    fig = go.Figure(data=[go.Scatter3d(
        x=points_vis[:, 0],
        y=points_vis[:, 1],
        z=points_vis[:, 2],
        mode='markers',
        marker=dict(
            size=size,
            color=colors_vis,
            colorscale='Viridis',
            colorbar=dict(title="Z (m)"),
            opacity=0.8
        )
    )])
    
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            aspectmode='data'
        ),
        width=900,
        height=700
    )
    
    fig.show()

# 원본 데이터 시각화
print("🔍 원본 포인트 클라우드")
visualize_pointcloud_3d(xyz, title="원본 포인트 클라우드 (Original)")

## 4. 아웃라이어 제거 (Statistical Outlier Removal)

바닥면 외의 물체(파이프, 장비, 작업자 등)로 인한 튀는 점들을 제거합니다.

In [ ]:
def remove_outliers_statistical(points, nb_neighbors=20, std_ratio=2.0):
    """
    Statistical Outlier Removal
    
    Parameters:
    - nb_neighbors: 이웃 포인트 수 (클수록 보수적)
    - std_ratio: 표준편차 배수 (작을수록 엄격하게 제거)
    """
    print(f"\n🧹 아웃라이어 제거 중...")
    print(f"   - 이웃 포인트 수: {nb_neighbors}")
    print(f"   - 표준편차 배수: {std_ratio}")
    
    # Open3D 포인트 클라우드 생성
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    # Statistical Outlier Removal
    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, 
                                              std_ratio=std_ratio)
    
    inlier_mask = np.zeros(len(points), dtype=bool)
    inlier_mask[ind] = True
    
    cleaned_points = points[inlier_mask]
    
    removed_count = len(points) - len(cleaned_points)
    print(f"\n   ✅ 제거된 아웃라이어: {removed_count:,}개 ({removed_count/len(points)*100:.2f}%)")
    print(f"   ✅ 남은 포인트: {len(cleaned_points):,}개")
    
    return cleaned_points, inlier_mask

In [ ]:
#@title 아웃라이어 제거 파라미터 설정 { run: "auto" }
NB_NEIGHBORS = 20  #@param {type:"slider", min:5, max:50, step:5}
STD_RATIO = 2.0  #@param {type:"slider", min:0.5, max:4.0, step:0.5}

xyz_cleaned, inlier_mask = remove_outliers_statistical(xyz, NB_NEIGHBORS, STD_RATIO)

In [ ]:
# 아웃라이어 제거 전/후 비교
print("\n🔍 아웃라이어 제거 전/후 비교")

fig = make_subplots(rows=1, cols=2, 
                    specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
                    subplot_titles=('원본 (Original)', '정제 후 (Cleaned)'))

# 샘플링
sample_size = 50000
idx_orig = np.random.choice(len(xyz), min(sample_size, len(xyz)), replace=False)
idx_clean = np.random.choice(len(xyz_cleaned), min(sample_size, len(xyz_cleaned)), replace=False)

fig.add_trace(
    go.Scatter3d(x=xyz[idx_orig, 0], y=xyz[idx_orig, 1], z=xyz[idx_orig, 2],
                 mode='markers', marker=dict(size=1, color=xyz[idx_orig, 2], colorscale='Viridis'),
                 name='Original'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter3d(x=xyz_cleaned[idx_clean, 0], y=xyz_cleaned[idx_clean, 1], z=xyz_cleaned[idx_clean, 2],
                 mode='markers', marker=dict(size=1, color=xyz_cleaned[idx_clean, 2], colorscale='Viridis'),
                 name='Cleaned'),
    row=1, col=2
)

fig.update_layout(height=600, width=1200, title_text="아웃라이어 제거 전/후 비교")
fig.show()

## 5. RANSAC을 이용한 기준 평면 피팅

바닥면의 기준이 되는 가상의 평면을 RANSAC 알고리즘으로 추정합니다.

평면 방정식: `ax + by + cz + d = 0`

In [ ]:
def fit_plane_ransac(points, distance_threshold=0.01, ransac_n=3, num_iterations=1000):
    """
    RANSAC을 이용한 평면 피팅
    
    Parameters:
    - distance_threshold: 인라이어 판정 거리 임계값 (m)
    - ransac_n: 평면 추정에 사용할 최소 포인트 수
    - num_iterations: RANSAC 반복 횟수
    """
    print(f"\n📐 RANSAC 평면 피팅 중...")
    print(f"   - 거리 임계값: {distance_threshold*1000:.1f} mm")
    print(f"   - 반복 횟수: {num_iterations}")
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    plane_model, inliers = pcd.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=ransac_n,
        num_iterations=num_iterations
    )
    
    [a, b, c, d] = plane_model
    print(f"\n   ✅ 평면 방정식: {a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0")
    print(f"   ✅ 평면 법선 벡터: ({a:.4f}, {b:.4f}, {c:.4f})")
    print(f"   ✅ 인라이어: {len(inliers):,}개 ({len(inliers)/len(points)*100:.1f}%)")
    
    return plane_model, inliers

In [ ]:
#@title RANSAC 파라미터 설정 { run: "auto" }
DISTANCE_THRESHOLD = 0.01  #@param {type:"number"}
NUM_ITERATIONS = 1000  #@param {type:"slider", min:100, max:5000, step:100}

plane_model, plane_inliers = fit_plane_ransac(xyz_cleaned, DISTANCE_THRESHOLD, num_iterations=NUM_ITERATIONS)

## 6. 평면으로부터의 거리(편차) 계산

In [ ]:
def calculate_distance_to_plane(points, plane_model):
    """
    각 포인트에서 평면까지의 거리 계산
    
    평면 방정식: ax + by + cz + d = 0
    거리 공식: |ax0 + by0 + cz0 + d| / sqrt(a² + b² + c²)
    """
    [a, b, c, d] = plane_model
    
    # 부호가 있는 거리 (평면 위/아래 구분)
    signed_distances = (a * points[:, 0] + b * points[:, 1] + c * points[:, 2] + d) / np.sqrt(a**2 + b**2 + c**2)
    abs_distances = np.abs(signed_distances)
    
    print(f"\n📏 평면으로부터의 거리 통계 (단위: mm):")
    print(f"   - 평균: {np.mean(abs_distances)*1000:.2f}")
    print(f"   - 중앙값: {np.median(abs_distances)*1000:.2f}")
    print(f"   - 표준편차: {np.std(abs_distances)*1000:.2f}")
    print(f"   - 최소: {np.min(abs_distances)*1000:.2f}")
    print(f"   - 최대: {np.max(abs_distances)*1000:.2f}")
    print(f"\n   백분위수:")
    for p in [50, 90, 95, 99]:
        print(f"   - {p}%: {np.percentile(abs_distances, p)*1000:.2f} mm")
    
    return signed_distances, abs_distances

signed_distances, abs_distances = calculate_distance_to_plane(xyz_cleaned, plane_model)

In [ ]:
# 거리 분포 히스토그램
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 절대 거리
axes[0].hist(abs_distances * 1000, bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=np.mean(abs_distances)*1000, color='red', linestyle='--', linewidth=2, 
                label=f'평균: {np.mean(abs_distances)*1000:.2f}mm')
axes[0].axvline(x=np.median(abs_distances)*1000, color='orange', linestyle='--', linewidth=2, 
                label=f'중앙값: {np.median(abs_distances)*1000:.2f}mm')
axes[0].set_xlabel('평면으로부터의 거리 (mm)', fontsize=12)
axes[0].set_ylabel('포인트 수', fontsize=12)
axes[0].set_title('거리 분포 (Absolute Distance)', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 부호 있는 거리
axes[1].hist(signed_distances * 1000, bins=100, edgecolor='black', alpha=0.7, color='coral')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=2, label='기준 평면')
axes[1].set_xlabel('거리 (mm) [+: 평면 위, -: 평면 아래]', fontsize=12)
axes[1].set_ylabel('포인트 수', fontsize=12)
axes[1].set_title('부호 있는 거리 분포 (Signed Distance)', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 평탄하지 않은 영역 하이라이트

In [ ]:
def highlight_uneven_areas(points, distances, threshold_mm=10):
    """
    평탄하지 않은 영역을 색상으로 하이라이트
    
    색상 스케일:
    - 초록: 매우 평탄 (< threshold * 0.5)
    - 노랑: 평탄 (< threshold)
    - 주황: 약간 불평탄 (< threshold * 2)
    - 빨강: 매우 불평탄 (>= threshold * 2)
    """
    threshold = threshold_mm / 1000
    abs_dist = np.abs(distances)
    
    colors = np.zeros((len(points), 3))
    
    for i, dist in enumerate(abs_dist):
        if dist <= threshold * 0.5:
            colors[i] = [0, 0.8, 0]  # 초록
        elif dist <= threshold:
            ratio = (dist - threshold * 0.5) / (threshold * 0.5)
            colors[i] = [ratio, 0.8, 0]  # 초록 -> 노랑
        elif dist <= threshold * 2:
            ratio = (dist - threshold) / threshold
            colors[i] = [1, 0.8 - ratio * 0.5, 0]  # 노랑 -> 주황
        else:
            colors[i] = [1, 0, 0]  # 빨강
    
    uneven_mask = abs_dist > threshold
    uneven_count = np.sum(uneven_mask)
    
    print(f"\n🎨 평탄성 분류 (임계값: {threshold_mm}mm):")
    print(f"   - 🟢 평탄 영역: {len(points) - uneven_count:,}개 ({(1 - uneven_count/len(points))*100:.1f}%)")
    print(f"   - 🔴 불평탄 영역: {uneven_count:,}개 ({uneven_count/len(points)*100:.1f}%)")
    
    return colors, uneven_mask

In [ ]:
#@title 평탄성 임계값 설정 (mm) { run: "auto" }
FLATNESS_THRESHOLD_MM = 10  #@param {type:"slider", min:1, max:50, step:1}

colors, uneven_mask = highlight_uneven_areas(xyz_cleaned, signed_distances, FLATNESS_THRESHOLD_MM)

In [ ]:
# 3D 시각화 (평탄성 하이라이트)
print("\n🔍 평탄성 하이라이트 3D 시각화")

sample_size = min(100000, len(xyz_cleaned))
idx = np.random.choice(len(xyz_cleaned), sample_size, replace=False)

fig = go.Figure(data=[go.Scatter3d(
    x=xyz_cleaned[idx, 0],
    y=xyz_cleaned[idx, 1],
    z=xyz_cleaned[idx, 2],
    mode='markers',
    marker=dict(
        size=2,
        color=['rgb({},{},{})'.format(int(c[0]*255), int(c[1]*255), int(c[2]*255)) for c in colors[idx]],
        opacity=0.8
    )
)])

fig.update_layout(
    title=f'바닥면 평탄성 분석 결과 (임계값: {FLATNESS_THRESHOLD_MM}mm)<br>'
          f'<span style="color:green">■</span> 평탄 | '
          f'<span style="color:yellow">■</span> 경계 | '
          f'<span style="color:red">■</span> 불평탄',
    scene=dict(
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        aspectmode='data'
    ),
    width=1000,
    height=800
)

fig.show()

## 8. 2D 히트맵 시각화

In [ ]:
def create_flatness_heatmap(points, distances, grid_size=100, threshold_mm=10):
    """2D 평탄성 히트맵 생성"""
    
    x, y = points[:, 0], points[:, 1]
    
    xi = np.linspace(x.min(), x.max(), grid_size)
    yi = np.linspace(y.min(), y.max(), grid_size)
    xi_grid, yi_grid = np.meshgrid(xi, yi)
    
    zi = griddata((x, y), np.abs(distances) * 1000, (xi_grid, yi_grid), method='linear')
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 히트맵 (연속)
    im1 = axes[0].contourf(xi_grid, yi_grid, zi, levels=50, cmap='RdYlGn_r')
    plt.colorbar(im1, ax=axes[0], label='거리 (mm)')
    axes[0].contour(xi_grid, yi_grid, zi, levels=[threshold_mm], colors='black', linewidths=2)
    axes[0].set_xlabel('X (m)')
    axes[0].set_ylabel('Y (m)')
    axes[0].set_title(f'평탄성 히트맵 (검은선: {threshold_mm}mm 임계값)')
    axes[0].set_aspect('equal')
    
    # 히트맵 (이진)
    binary_map = np.where(zi > threshold_mm, 1, 0)
    axes[1].contourf(xi_grid, yi_grid, binary_map, levels=[-0.5, 0.5, 1.5], colors=['green', 'red'], alpha=0.7)
    axes[1].set_xlabel('X (m)')
    axes[1].set_ylabel('Y (m)')
    axes[1].set_title(f'평탄/불평탄 영역 (임계값: {threshold_mm}mm)\n🟢 평탄 | 🔴 불평탄')
    axes[1].set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    return xi_grid, yi_grid, zi

xi_grid, yi_grid, zi = create_flatness_heatmap(xyz_cleaned, signed_distances, 
                                                grid_size=150, 
                                                threshold_mm=FLATNESS_THRESHOLD_MM)

## 9. 분석 결과 요약

In [ ]:
def generate_analysis_report(points, distances, threshold_mm, plane_model):
    """분석 결과 요약 리포트"""
    abs_dist = np.abs(distances) * 1000
    
    uneven_count = np.sum(abs_dist > threshold_mm)
    uneven_pct = uneven_count / len(points) * 100
    
    print("=" * 60)
    print("          🏗️ 바닥면 평탄성 분석 결과 요약")
    print("=" * 60)
    
    print(f"\n📊 데이터 정보")
    print(f"   - 총 포인트 수: {len(points):,}")
    print(f"   - 분석 영역: {points[:,0].max()-points[:,0].min():.2f}m × {points[:,1].max()-points[:,1].min():.2f}m")
    
    print(f"\n📐 기준 평면 (RANSAC)")
    [a, b, c, d] = plane_model
    print(f"   - 방정식: {a:.4f}x + {b:.4f}y + {c:.4f}z + {d:.4f} = 0")
    
    print(f"\n📏 평탄성 통계 (mm)")
    print(f"   - 평균 편차: {np.mean(abs_dist):.2f}")
    print(f"   - 중앙값: {np.median(abs_dist):.2f}")
    print(f"   - 표준편차: {np.std(abs_dist):.2f}")
    print(f"   - 최대 편차: {np.max(abs_dist):.2f}")
    
    print(f"\n🎯 평탄성 평가 (임계값: {threshold_mm}mm)")
    print(f"   - 🟢 평탄 영역: {len(points) - uneven_count:,}개 ({100 - uneven_pct:.1f}%)")
    print(f"   - 🔴 불평탄 영역: {uneven_count:,}개 ({uneven_pct:.1f}%)")
    
    print(f"\n📋 종합 등급: ", end="")
    if uneven_pct <= 5:
        print("⭐ 우수 (Excellent)")
        grade = "우수"
    elif uneven_pct <= 15:
        print("✅ 양호 (Good)")
        grade = "양호"
    elif uneven_pct <= 30:
        print("⚠️ 보통 (Fair)")
        grade = "보통"
    else:
        print("🔴 불량 (Poor)")
        grade = "불량"
    
    print("=" * 60)
    
    return {
        'total_points': len(points),
        'mean_deviation_mm': np.mean(abs_dist),
        'max_deviation_mm': np.max(abs_dist),
        'uneven_percentage': uneven_pct,
        'grade': grade
    }

report = generate_analysis_report(xyz_cleaned, signed_distances, FLATNESS_THRESHOLD_MM, plane_model)

## 10. 결과 내보내기

In [ ]:
def export_results(points, distances, colors, filename_prefix="floor_analysis"):
    """분석 결과를 CSV로 내보내기"""
    
    df = pd.DataFrame({
        'X': points[:, 0],
        'Y': points[:, 1],
        'Z': points[:, 2],
        'Distance_mm': np.abs(distances) * 1000,
        'Signed_Distance_mm': distances * 1000,
        'R': (colors[:, 0] * 255).astype(int),
        'G': (colors[:, 1] * 255).astype(int),
        'B': (colors[:, 2] * 255).astype(int),
        'Is_Uneven': np.abs(distances) * 1000 > FLATNESS_THRESHOLD_MM
    })
    
    csv_filename = f"{filename_prefix}_results.csv"
    df.to_csv(csv_filename, index=False)
    print(f"✅ 결과 저장됨: {csv_filename}")
    
    files.download(csv_filename)
    
    return df

# 결과 내보내기
print("\n📥 결과를 CSV로 내보내기")
export_df = export_results(xyz_cleaned, signed_distances, colors)

## 11. (선택) 불평탄 영역 클러스터링

In [ ]:
def find_defect_clusters(points, distances, threshold_mm=10, eps=0.1, min_samples=50):
    """불평탄 영역을 클러스터링하여 결함 위치 찾기"""
    
    uneven_mask = np.abs(distances) * 1000 > threshold_mm
    uneven_points = points[uneven_mask]
    uneven_distances = distances[uneven_mask]
    
    if len(uneven_points) < min_samples:
        print("⚠️ 불평탄 포인트가 충분하지 않습니다.")
        return []
    
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(uneven_points)
    labels = clustering.labels_
    
    clusters = []
    unique_labels = set(labels) - {-1}
    
    print(f"\n🔍 발견된 불평탄 클러스터: {len(unique_labels)}개")
    
    for label in sorted(unique_labels):
        mask = labels == label
        cluster_points = uneven_points[mask]
        cluster_distances = uneven_distances[mask]
        
        cluster_info = {
            'label': label,
            'size': len(cluster_points),
            'center': np.mean(cluster_points, axis=0),
            'max_deviation_mm': np.max(np.abs(cluster_distances)) * 1000,
            'mean_deviation_mm': np.mean(np.abs(cluster_distances)) * 1000
        }
        clusters.append(cluster_info)
        
        print(f"   클러스터 #{label}: {cluster_info['size']:,}개 포인트")
        print(f"      위치: ({cluster_info['center'][0]:.2f}, {cluster_info['center'][1]:.2f})")
        print(f"      최대 편차: {cluster_info['max_deviation_mm']:.1f}mm")
    
    return clusters

clusters = find_defect_clusters(xyz_cleaned, signed_distances, 
                                 threshold_mm=FLATNESS_THRESHOLD_MM,
                                 eps=0.1, min_samples=50)

---
## 📝 사용 방법 요약

1. **파일 업로드**: 포인트 클라우드 파일 (.txt, .xyz) 업로드
2. **파라미터 조정**:
   - `NB_NEIGHBORS`, `STD_RATIO`: 아웃라이어 제거 강도
   - `DISTANCE_THRESHOLD`: RANSAC 인라이어 임계값 (m)
   - `FLATNESS_THRESHOLD_MM`: 평탄/불평탄 판정 임계값 (mm)
3. **결과 확인**: 3D 시각화, 2D 히트맵, 통계 분석
4. **내보내기**: 결과 CSV 파일 다운로드